In [1]:
import os
from loader import read_dictionary, read_sentences
from candidates import BKTree, CandidateGenerator, SymSpell
from ngram_model import KNgramModel
from corrections import SpellCorrector
from spylls.hunspell import Dictionary
import random
import math
from tqdm import tqdm
import copy



In [2]:
language = "es"
misfit_file = True

data_dir = "./data"

hunspell_dir = os.path.join(data_dir, "hunspell")
hunspell_dict_dir = os.path.join(hunspell_dir, "es_ES")
dict_path = os.path.join(hunspell_dir, "es_ES_unmunched_words.txt")

sentences_path = os.path.join(data_dir, f"{language}_sentences.txt")

model_dir = "./models"

bk_tree_path = os.path.join(model_dir, "bk_tree.pkl")
sym_spell_path = os.path.join(model_dir, "sym_spell.pkl")

forward_lm_path = os.path.join(model_dir, "forward_lm.pkl")
backward_lm_path = os.path.join(model_dir, "backward_lm.pkl")


In [3]:
lexicon = Dictionary.from_files(hunspell_dict_dir)


In [4]:
# tree = BKTree.load(bk_tree_path)


In [5]:
# result = tree.search("coch", 2)
# tree_set = {word for word, _ in result}
# print(tree_set)


In [6]:
sym_spell = SymSpell.load(sym_spell_path)


In [7]:
result = sym_spell.search("coch", 2)
sym_set = {word for word, _ in result}
print(sym_set)


{'coz', 'acocha', 'rocé', 'cocos', 'colo', 'corcha', 'cucó', 'ceca', 'cok', 'cacé', 'cace', 'cucho', 'cochea', 'roche', 'corcho', 'coge', 'cocí', 'choco', 'colché', 'cocho', 'coceo', 'coces', 'copa', 'loche', 'tocho', 'foca', 'acocho', 'copó', 'rocha', 'coca', 'toco', 'pocho', 'cocha', 'toca', 'concho', 'cocas', 'cloc', 'noche', 'oh', 'mochó', 'colé', 'gocé', 'codo', 'chocho', 'acoche', 'cota', 'colcha', 'cocer', 'doce', 'corco', 'corchó', 'cocheé', 'cocad', 'coco', 'copo', 'doca', 'loco', 'con', 'pocha', 'boca', 'corca', 'cosco', 'cosí', 'cote', 'caco', 'roca', 'cocea', 'chocha', 'cojo', 'rocho', 'cope', 'cole', 'cuco', 'cache', 'coches', 'moche', 'cose', 'cocó', 'cor', 'moco', 'colche', 'toce', 'coló', 'cuchó', 'locha', 'coció', 'rock', 'mocho', 'moché', 'concha', 'tocé', 'cocan', 'foco', 'acochó', 'tocó', 'oca', 'boché', 'cogí', 'coceó', 'clocó', 'copé', 'moca', 'coto', 'poca', 'hocé', 'coas', 'colchó', 'zoca', 'corche', 'comí', 'zoco', 'cocee', 'cuche', 'coño', 'coche', 'colcho', '

In [8]:
# print(len(tree_set ^ sym_set))


In [9]:
forward_lm = KNgramModel.load(forward_lm_path)


In [10]:
# dict = read_dictionary(dict_path)
# vocab = forward_lm.vocab

# covered = vocab & dict

# print(f"LM vocab size: {len(vocab):,}")
# print(f"Dictionary size: {len(dict):,}")
# print(f"Covered words: {len(covered):,}")
# print(f"Coverage: {100 * len(covered) / len(vocab):.2f}%")


In [11]:
backward_lm = KNgramModel.load(backward_lm_path)


In [36]:
spell_corrector = SpellCorrector(
	lexicon=lexicon,
	candidate_gen=sym_spell,
	forward_lm=forward_lm,
	backward_lm=backward_lm,
	max_distance=2
)


In [67]:
text = "Holaaaa buenas, soy {{nombre}}, acabo de llegar y estoy toabía un poco perdidde, y tú?"

spell_corrector.correct_text(
    text=text,
    alpha=0.0,
    beta=0.7
)


'Hola buenas, soy {{nombre}}, acabo de llegar y estoy todavía un poco perdido, y tú?'

In [ ]:
print(spell_corrector.correct_word(
	["yo", "conduzco", "el", "cocheee", "al", "trabajo"],
	3,
alpha=0.0, beta=0.5))

# escrivir -> escribir
print(spell_corrector.correct_word(
	["me", "gusta", "escrivir", "poemas"],
	2,
alpha=0.0, beta=0.5
))

# jente -> gente
print(spell_corrector.correct_word(
	["mucha", "jente", "vino", "a", "la", "fiesta"],
	1,
alpha=0.0, beta=0.5
))

# havía -> había
print(spell_corrector.correct_word(
	["no", "havía", "nadie", "en", "casa"],
	1,
alpha=0.0, beta=0.0
))

# avion -> avión
print(spell_corrector.correct_word(
	["el", "avion", "aterrizó", "a", "tiempo"],
	1,
alpha=0.0, beta=0.0
))

# univercidad -> universidad
print(spell_corrector.correct_word(
	["estudio", "en", "la", "univercidad", "de", "madrid"],
	3,
alpha=0.0, beta=0.0
))

# felis -> feliz
print(spell_corrector.correct_word(
	["estoy", "muy", "felis", "hoy"],
	2,
alpha=0.0, beta=0.0
))

# acia -> hacía
print(spell_corrector.correct_word(
	["ella", "acia", "los", "deberes", "cada", "tarde"],
	1,
alpha=0.0, beta=0.0
))

# comienso -> comienzo
print(spell_corrector.correct_word(
	["el", "comienso", "del", "curso", "fue", "difícil"],
	1,
alpha=0.0, beta=0.0
))



In [5]:
sentences = read_sentences(sentences_path)


In [8]:
class SpellCorrectionEvaluator:
	LETTERS = set("abcdefghijklmnopqxyzáéíóúüñ")

	def __init__(self, lexicon: Dictionary, candidate_gen: CandidateGenerator, order: int = 3, discount: float = 0.75, unk_threshold: int = 2, max_distance: int = 2, train_ratio: float = 0.9, seed: int = 42):
		self.lexicon = lexicon
		self.candidate_gen = candidate_gen

		self.order = order
		self.unk_threshold = unk_threshold
		self.discount = discount
		
		self.max_distance = max_distance

		self.train_ratio = train_ratio

		self.forward_lm = None
		self.backward_lm = None
		self.spell_corrector = None

		random.seed(seed)

	def train_test_split(self, sentences: list[list[str]]):
		sentences = sentences.copy()
		random.shuffle(sentences)

		split = math.ceil(self.train_ratio * len(sentences))

		train = sentences[:split]
		test = sentences[split:]

		return train, test

	def train_models(self, train_sentences: list[list[str]]):
		self.forward_lm = KNgramModel(
			order=self.order,
			unk_threshold=self.unk_threshold,
			discount=self.discount
		)
		self.forward_lm.train(train_sentences)

		reversed_train = [
			list(reversed(sentence))
			for sentence in train_sentences
		]

		self.backward_lm = KNgramModel(
			order=self.order,
			unk_threshold=self.unk_threshold,
			discount=self.discount
		)
		self.backward_lm.train(reversed_train)

		self.spell_corrector = SpellCorrector(
			lexicon=self.lexicon,
			candidate_gen=self.candidate_gen,
			forward_lm=self.forward_lm,
			backward_lm=self.backward_lm,
			max_distance=self.max_distance,
		)

	def random_typo(self, word: str, k: int = 2) -> str:
		chars = list(word)
		for _ in range(k):
			n = len(chars)
			ops = ["insert"]
			if n > 0:
				ops.append("delete")
				ops.append("substitute")
			if n > 1:
				ops.append("transpose")

			op = random.choice(ops)

			if op == "delete" and n > 0:
				i = random.randrange(n)
				del chars[i]

			elif op == "insert":
				c = random.choice(list(self.LETTERS))
				i = random.randrange(n + 1)
				chars.insert(i, c)

			elif op == "substitute":
				i = random.randrange(n)
				chars[i] = random.choice(list(self.LETTERS))

			elif op == "transpose" and n > 1:
				i = random.randrange(n - 1)
				chars[i], chars[i + 1] = chars[i + 1], chars[i]

		result = "".join(chars)
		return result if result != word else self.random_typo(word, k)

	def generate_error_word(self, word: str, edit_distance: int = 2):
		max_attempts = 10
		for _ in range(max_attempts):
			typo = self.random_typo(word, edit_distance)
			if not self.lexicon.lookup(typo):
				return typo
		return typo

	def generate_errors(self, data: list[list[str]], errors_per_sentence: int = 1) -> tuple[list[list[str]], list[list[int]]]:
		corrupted = copy.deepcopy(data)
		error_locations = []

		for sentence in tqdm(corrupted, desc="Generating errors"):
			n_words = len(sentence)

			indices = random.sample(
				range(n_words),
				min(errors_per_sentence, n_words)
			)

			error_locations.append(indices)

			for idx in indices:
				edit_dist = random.randint(1, self.max_distance)
				sentence[idx] = self.generate_error_word(
					sentence[idx],
					edit_dist
				)

		return corrupted, error_locations
	
	def predict(self, corrupted_data: list[list[str]], error_locations: list[list[int]], alpha: float = 1.0, beta: float = 0.5):
		predictions = []

		for sentence, indices in tqdm(zip(corrupted_data, error_locations), total=len(corrupted_data), desc="Correcting"):
			pred = sentence.copy()

			for idx in indices:
				pred[idx] = self.spell_corrector.correct_word(
					sentence=pred,
					position=idx,
					alpha=alpha,
					beta=beta
				)

			predictions.append(pred)

		return predictions

	def evaluate(self, test_data: list[list[str]], predictions: list[list[str]], error_locations: list[list[int]]):
		total_errors = 0
		corrected_errors = 0
		sentence_hits = 0

		for pred, truth, indices in tqdm(zip(predictions, test_data, error_locations), desc="Evaluating accuracy"):
			sentence_ok = True

			for idx in indices:
				total_errors += 1

				if pred[idx] == truth[idx]:
					corrected_errors += 1
				else:
					sentence_ok = False

			if sentence_ok:
				sentence_hits += 1

		return {
			"error_accuracy": corrected_errors / total_errors,
			"sentence_accuracy": sentence_hits / len(test_data),
			"corrected_errors": corrected_errors,
			"total_errors": total_errors,
		}
	
	def coverage(self, test_data: list[list[str]], errors_per_sentence: int = 1):
		corrupted, locations = self.generate_errors(test_data, errors_per_sentence)
		
		total_errors = 0
		true_in_candidates = 0
		
		for corrupt_sent, truth, indices in tqdm(zip(corrupted, test_data, locations), total=len(corrupted), desc="Coverage"):
			for idx in indices:
				typo = corrupt_sent[idx]
				true_word = truth[idx]
				total_errors += 1

				candidates = self.candidate_gen.search(typo, self.max_distance)
				candidate_words = {cand for cand, _ in candidates}
				if true_word.lower() in candidate_words:
					true_in_candidates += 1
		
		return {
			"coverage": true_in_candidates / total_errors,
			"true_in_candidates": true_in_candidates,
			"total_errors": total_errors
		}
	

In [9]:
evaluator = SpellCorrectionEvaluator(
	lexicon=lexicon,
	candidate_gen=sym_spell,
    train_ratio=0.95,
    order=3,
    unk_threshold=1,
)


In [10]:
evaluator.random_typo("hola")


'hala'

In [11]:
train, test = evaluator.train_test_split(sentences)

print(f"Train: {len(train):,}")
print(f"Test : {len(test):,}")


Train: 4,132,479
Test : 217,498


In [12]:
test_subset = test[:10_000]
print(test_subset[:3])


[['cómo', 'está', 'la', 'sra', 'buchanan'], ['que', 'está', 'muerta', 'mamón'], ['hemos', 'ahorrado', 'cada', 'moneda', 'y', 'tu']]


In [13]:
evaluator.coverage(test_subset)


Coverage: 100%|██████████| 10000/10000 [00:35<00:00, 282.69it/s]


{'coverage': 0.9203, 'true_in_candidates': 9203, 'total_errors': 10000}

In [14]:
evaluator.train_models(train)


Training: 100%|██████████| 4132479/4132479 [02:31<00:00, 27233.23it/s]


In [15]:
test_subset = test[:3_000]


In [16]:
corrupted, locations = evaluator.generate_errors(
    test_subset,
    errors_per_sentence=1
)

print(corrupted[:3])


Generating errors: 100%|██████████| 3000/3000 [00:00<00:00, 6301.41it/s]

[['ómok', 'está', 'la', 'sra', 'buchanan'], ['uqe', 'está', 'muerta', 'mamón'], ['hemos', 'aohrrdo', 'cada', 'moneda', 'y', 'tu']]


In [17]:
predictions = evaluator.predict(
	corrupted[:3],
	locations,
    alpha=1.2,
    beta=0.2
)

print(predictions)


Correcting: 100%|██████████| 3/3 [00:00<00:00, 16.48it/s]

[['cómo', 'está', 'la', 'sra', 'buchanan'], ['que', 'está', 'muerta', 'mamón'], ['hemos', 'bohordo', 'cada', 'moneda', 'y', 'tu']]


In [18]:
predictions = evaluator.predict(
	corrupted,
	locations,
    alpha=1.2,
    beta=0.2
)

metrics = evaluator.evaluate(test_subset, predictions, locations)
print(metrics)



Correcting:   0%|          | 0/3000 [00:00<?, ?it/s]

Correcting: 100%|██████████| 3000/3000 [02:40<00:00, 18.70it/s]
Evaluating accuracy: 3000it [00:00, 428456.55it/s]

{'error_accuracy': 0.761, 'sentence_accuracy': 0.761, 'corrected_errors': 2283, 'total_errors': 3000}


In [19]:
print(metrics)


{'error_accuracy': 0.761, 'sentence_accuracy': 0.761, 'corrected_errors': 2283, 'total_errors': 3000}
